# Week 18 DE: RAG - Part 2 for Data Engineers

## Measuring and Improving the Week 17 Pipeline Ops KB

## Learning Objectives

By the end of this session, you will be able to:
1. **Explain why runbooks require larger chunks** than fraud policies and measure the
   retrieval difference across three chunking configs
2. **Add Cohere Rerank 3.5** on top of the Week 17 `runbook_retriever_agent` and
   quantify the precision lift on ops queries
3. **Evaluate RAG outputs with RAGAS v0.4** (faithfulness, answer_relevancy) using
   Bedrock Claude Haiku as the judge - no external API keys
4. **Build a `PipelineOpsAdvisor`** that extends the Week 17 `ops_supervisor` with
   the optimized retrieval config and RAGAS-validated quality

## Prerequisites

- Completed Week 17 DE optional (`week_17_optional_data_engineer.ipynb`)
- `STRANDS_DE_KNOWLEDGE_BASE_ID` from Week 17 (same class KB)
- Watched pre-class videos on chunking, reranking, RAGAS

## Session Format (~2 hours)

| Section | Duration | Type |
|---------|----------|------|
| Section 0: Setup and Week 17 DE Recap | 10 min | Code |
| Section 1: Chunking for Data Engineering | 30 min | Theory + Demo + Lab 1 |
| Section 2: Reranking the RunbookRetriever | 28 min | Theory + Demo + Lab 2 |
| Section 3: RAGAS + PipelineOpsAdvisor | 28 min | Theory + Demo + Lab 3 |
| Wrap-up and Homework | 5 min | Markdown |

## The Story So Far

In Week 17 DE you built a Pipeline Ops Supervisor that routes on-call queries to three
KB-backed specialists: CatalogRetriever, RunbookRetriever, GovernanceRetriever. It WORKS.
But when an on-call engineer asks "How do I recover from a schema validation failure in
the nightly_customers_etl?" and the runbook chunk is truncated mid-procedure, the agent
gives an incomplete answer.

This week you pull the same three levers the ML engineer pulls on the fraud KB - but for
your runbook and catalog corpus:

- **Chunking**: why runbooks need larger chunks than policy documents
- **Reranking**: Cohere Rerank 3.5 on runbook queries for precision lift
- **RAGAS**: faithfulness + answer_relevancy to turn "looks right" into numbers

The take-home is not a new supervisor. It is the Week 17 supervisor with measured,
defensible retrieval quality under it.

## No GPU Needed

All work is API-based through Amazon Bedrock.

# Section 0: Environment Setup and Week 17 DE Recap

Same stack as Week 17 DE main, plus three new libraries for chunking experiments and
evaluation:

- `langchain` + `langchain-aws` - local chunking experiments and the Bedrock bridge RAGAS needs
- `langchain-community` - BedrockEmbeddings wrapper for RAGAS
- `ragas==0.4.3` - the evaluation framework, with Bedrock Claude Haiku 3 as judge

Nothing new for reranking - the Bedrock Rerank API is already reachable through
`boto3.client('bedrock-agent-runtime').rerank(...)`.

AWS credentials come from your SageMaker execution role - same as Week 17 DE.
No `getpass`, no API keys.

In [ ]:
# Install required libraries. Versions are lower-bound only so pip resolves
# quickly from cache. sagemaker is pinned to v2.x because v3 removed
# get_execution_role() from the top-level namespace.

%pip install -q \
    "sagemaker>=2.200,<3" \
    "strands-agents>=1.37" \
    "strands-agents-tools[mem0-memory]>=0.2.10" \
    "boto3>=1.35" \
    "langchain>=0.3" \
    "langchain-aws>=0.2" \
    "langchain-community>=0.3" \
    "ragas>=0.4" \
    "datasets>=2.18" \
    "faiss-cpu>=1.9" \
    "rank_bm25>=0.2.2" \
    "opensearch-py>=2.4"

print("\nPackages installed. If this was your first install, RESTART THE KERNEL before the next cell.")

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
# IMPORTANT: strands_tools.retrieve reads KNOWLEDGE_BASE_ID at IMPORT TIME.
# We set the DE KB id BEFORE the strands import.

import os
import json
import time
import boto3
import sagemaker
import pandas as pd
from sagemaker import get_execution_role
from importlib.metadata import version as pkg_version

# =============================================================================
# SET strands_tools.retrieve ENVIRONMENT VARIABLES (before import)
# =============================================================================
STRANDS_DE_KNOWLEDGE_BASE_ID = (
    os.environ.get("STRANDS_DE_KNOWLEDGE_BASE_ID")
    or "HWLCYXAEGP"
)
os.environ["STRANDS_DE_KNOWLEDGE_BASE_ID"]     = STRANDS_DE_KNOWLEDGE_BASE_ID
os.environ["KNOWLEDGE_BASE_ID"]                = STRANDS_DE_KNOWLEDGE_BASE_ID
os.environ["MIN_SCORE"]                        = os.environ.get("MIN_SCORE", "0.2")
os.environ["RETRIEVE_ENABLE_METADATA_DEFAULT"] = "true"

# =============================================================================
# STRANDS + LANGCHAIN IMPORTS (after env vars)
# =============================================================================
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import retrieve

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_aws import BedrockEmbeddings, ChatBedrockConverse
from langchain_core.documents import Document

from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics import faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

for pkg in ["strands-agents", "boto3", "langchain", "langchain-aws", "ragas", "faiss-cpu", "numpy"]:
    try:
        print(f"  {pkg:22s} {pkg_version(pkg)}")
    except Exception:
        print(f"  {pkg:22s} (not installed)")

# =============================================================================
# SAGEMAKER SESSION + EXECUTION ROLE (same as Week 15/16/17)
# =============================================================================
sess = sagemaker.Session()
role = get_execution_role()
AWS_REGION = sess.boto_region_name

os.environ["AWS_REGION"]         = AWS_REGION
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION

print(f"\nSageMaker execution role: {role.split('/')[-1]}")
print(f"AWS Region:               {AWS_REGION}")

# =============================================================================
# MODEL CONFIGURATION (same as Week 17 DE)
# =============================================================================
MODEL_ID       = "us.anthropic.claude-3-haiku-20240307-v1:0"
EMBED_MODEL_ID = "amazon.titan-embed-text-v2:0"

llm                = BedrockModel(model_id=MODEL_ID, region_name=AWS_REGION)
langchain_llm      = ChatBedrockConverse(model=MODEL_ID, region_name=AWS_REGION)
bedrock_embeddings = BedrockEmbeddings(model_id=EMBED_MODEL_ID, region_name=AWS_REGION)

evaluator_llm        = LangchainLLMWrapper(langchain_llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(bedrock_embeddings)

print(f"\nLLM:      {MODEL_ID}")
print(f"Embed:    {EMBED_MODEL_ID}")
print(f"Reranker: Claude Haiku listwise (same model, no marketplace required)")

# =============================================================================
# PRE-FLIGHT PROBES
# =============================================================================
bedrock_runtime       = boto3.client("bedrock-runtime",       region_name=AWS_REGION)
bedrock_agent         = boto3.client("bedrock-agent",         region_name=AWS_REGION)
bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=AWS_REGION)

# Probe 1: LLM access
try:
    probe = bedrock_runtime.converse(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": [{"text": "ping"}]}],
        inferenceConfig={"maxTokens": 10, "temperature": 0},
    )
    print(f"\nLLM probe OK:    {probe['output']['message']['content'][0]['text']!r}")
except Exception as e:
    print(f"\nLLM probe FAILED: {e}")
    print(f"Ask your instructor to enable Bedrock access for {MODEL_ID}.")
    raise

# Probe 2: DE Knowledge Base
try:
    kb_info = bedrock_agent.get_knowledge_base(knowledgeBaseId=STRANDS_DE_KNOWLEDGE_BASE_ID)
    print(f"KB probe OK:     {STRANDS_DE_KNOWLEDGE_BASE_ID} ({kb_info['knowledgeBase']['name']})")
except Exception as e:
    print(f"KB probe FAILED: {e}")
    print("Ask your instructor for the correct STRANDS_DE_KNOWLEDGE_BASE_ID.")
    raise

print("\nEnvironment ready.")

In [ ]:
# =============================================================================
# WEEK 17 DE RECAP - baseline retrieve_with_filter + runbook agent + corpus
# =============================================================================
# Re-declared inline so this notebook is self-contained.

def retrieve_with_filter(query: str, doc_type: str, num_results: int = 3) -> list:
    """Retrieve chunks filtered by doc_type from the DE KB."""
    resp = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=STRANDS_DE_KNOWLEDGE_BASE_ID,
        retrievalQuery={"text": query},
        retrievalConfiguration={
            "vectorSearchConfiguration": {
                "numberOfResults": num_results,
                "filter": {"equals": {"key": "doc_type", "value": doc_type}},
            },
        },
    )
    return [
        {
            "text":   r.get("content", {}).get("text", ""),
            "score":  r.get("score", 0.0),
            "source": r.get("location", {}).get("s3Location", {}).get("uri", ""),
        }
        for r in resp.get("retrievalResults", [])
    ]


@tool
def retrieve_runbook(query: str) -> str:
    """Retrieve runbook chunks for on-call recovery and troubleshooting.

    Args:
        query: The ops scenario or recovery question.
    """
    hits = retrieve_with_filter(query, doc_type="runbook", num_results=3)
    return json.dumps({"matches": hits, "count": len(hits)}, indent=2)


runbook_retriever_agent = Agent(
    model=llm,
    tools=[retrieve_runbook],
    system_prompt=(
        "You are a data platform on-call specialist. You have access only to "
        "recovery runbooks. Answer only from retrieved runbook content. "
        "If the runbook does not cover the issue, say so."
    ),
    callback_handler=None,
)

baseline_ops_supervisor = Agent(
    model=llm,
    tools=[retrieve_runbook],
    system_prompt=(
        "You are the pipeline ops supervisor. For pipeline failure questions, "
        "retrieve the relevant runbook. For schema or lineage questions, "
        "retrieve the catalog. Route to the right specialist."
    ),
    callback_handler=None,
)

# =============================================================================
# PIPELINE OPS CORPUS - inline text for chunking experiments
# =============================================================================
# Same content type as the DE KB (runbooks + catalog + governance) but declared
# inline so we can freely experiment with chunk size.
# Note the runbooks use structured sections and bash code blocks.
# Fixed-size 300-token chunking will split these; config C (1500/200) keeps them whole.

PIPELINE_OPS_CORPUS = [
    ("runbook_schema_validation.md",
     "# Schema Validation Failure - Recovery Runbook\n\n"
     "## Symptoms\n"
     "The nightly_customers_etl job fails at the VALIDATE step with "
     "`SchemaValidationError: Column 'annual_income' expected FLOAT, got STRING`. "
     "The pipeline halts before writing to S3. Downstream jobs (reporting_tables, "
     "fraud_detection_model_features) are blocked.\n\n"
     "## Root Cause\n"
     "Upstream source (Salesforce CRM export) changed the column type without notice. "
     "This typically happens after a Salesforce release on the 3rd Tuesday of each month.\n\n"
     "## Recovery Steps\n"
     "```bash\n"
     "# Step 1: identify the bad column\n"
     "aws s3 cp s3://data-platform-raw/customers/latest/ /tmp/sample.parquet\n"
     "python3 -c \"import pandas as pd; df=pd.read_parquet('/tmp/sample.parquet'); "
     "print(df.dtypes)\"\n\n"
     "# Step 2: cast and re-run\n"
     "airflow dags backfill nightly_customers_etl --start-date 2026-04-15 "
     "--end-date 2026-04-15\n"
     "```\n\n"
     "## Escalation\n"
     "If the cast fails or the schema change is permanent, open a Jira ticket to the "
     "Data Contracts team (SLA: 4h response). Do not modify the pipeline schema without "
     "a Data Contracts review."),
    ("runbook_partition_missing.md",
     "# Missing Partition - Recovery Runbook\n\n"
     "## Symptoms\n"
     "Downstream queries fail with `S3 path not found: s3://datalake/customers/"
     "year=2026/month=04/day=16/`. The daily partition was not written.\n\n"
     "## Root Cause\n"
     "Usually caused by: (1) the ETL job failed silently, (2) S3 event notification "
     "did not trigger Glue crawler, (3) partition registration in Glue Data Catalog "
     "timed out.\n\n"
     "## Recovery Steps\n"
     "```bash\n"
     "# Check if data exists but partition is missing from catalog\n"
     "aws s3 ls s3://datalake/customers/year=2026/month=04/day=16/ --recursive\n\n"
     "# If data exists, add partition manually\n"
     "aws glue batch-create-partition --database-name prod_datalake "
     "--table-name customers --partition-input-list file://partition_spec.json\n"
     "```\n\n"
     "## SLA Impact\n"
     "This partition is on the critical path for the 6:00 AM reporting SLA. "
     "If not resolved by 5:30 AM, escalate to the on-call data engineer."),
    ("runbook_stale_data.md",
     "# Stale Data Alert - Recovery Runbook\n\n"
     "## Symptoms\n"
     "Monitoring alert: `DATA_FRESHNESS_BREACH - customers dataset last updated 26h ago "
     "(SLA: 24h)`. The fraud detection model is using yesterday's feature data.\n\n"
     "## Recovery Steps\n"
     "1. Check the Airflow UI for nightly_customers_etl - did it run? Did it succeed?\n"
     "2. If the DAG ran but the output is stale, check S3 for partial writes:\n"
     "   `aws s3 ls s3://datalake/customers/year=2026/month=04/day=16/ | wc -l`\n"
     "3. Re-trigger the DAG: `airflow dags trigger nightly_customers_etl`\n"
     "4. If re-trigger fails, fall back to the last known good partition.\n\n"
     "## Escalation SLA\n"
     "Stale data in the fraud feature store must be resolved within 2 hours of detection. "
     "Page the ML Platform team if the re-trigger does not complete within 45 minutes."),
    ("catalog_customers.md",
     "## Dataset Card: customers\n\n"
     "**Owner**: Data Platform Team\n"
     "**SLA**: Updated daily by 6:00 AM. 24-hour freshness guarantee.\n"
     "**Schema**: customer_id (STRING, PK), full_name (STRING), email (STRING, PII), "
     "annual_income (FLOAT), account_open_date (DATE), risk_score (FLOAT, derived)\n"
     "**Lineage**: Source: Salesforce CRM -> nightly_customers_etl -> "
     "s3://datalake/customers/ -> Glue Data Catalog -> fraud_detection_model_features\n"
     "**PII**: email, full_name. Masked in dev/staging environments.\n"
     "**Retention**: 7 years per BSA compliance requirements."),
    ("catalog_transactions.md",
     "## Dataset Card: transactions\n\n"
     "**Owner**: Payments Team\n"
     "**SLA**: Updated hourly. 2-hour freshness guarantee.\n"
     "**Schema**: txn_id (STRING, PK), customer_id (STRING, FK), merchant_id (STRING), "
     "amount (FLOAT), currency (STRING), mcc (STRING), timestamp (TIMESTAMP), "
     "status (STRING: APPROVED/DECLINED/PENDING)\n"
     "**Lineage**: CDC from payments_db -> Kafka -> transactions_stream_etl -> "
     "s3://datalake/transactions/ -> fraud_detection_model_features\n"
     "**PII**: None. customer_id is pseudonymized.\n"
     "**Retention**: 5 years."),
    ("governance_sla_policy.md",
     "## Data Platform SLA Policy\n\n"
     "**Critical pipelines** (must resolve within 2h): nightly_customers_etl, "
     "transactions_stream_etl, fraud_detection_model_features\n\n"
     "**Non-critical pipelines** (must resolve within 8h): reporting_tables, "
     "ml_feature_store_refresh, user_sessions_aggregation\n\n"
     "**Escalation path**: On-call DE -> Data Platform Lead -> VP of Engineering\n\n"
     "**Breach reporting**: Any SLA breach affecting fraud detection must be reported "
     "to the Risk Operations team within 1 hour of detection and to the CISO within 4h."),
    ("governance_pii_policy.md",
     "## PII Handling Policy\n\n"
     "**Classification**: email, full_name, phone, SSN, date_of_birth are PII.\n"
     "**Masking**: All PII columns are masked in dev and staging via the data_masking_etl "
     "job. Production access requires Data Governance approval.\n"
     "**Retention**: PII data retained 7 years (BSA), then purged via gdpr_purge_job.\n"
     "**Breach protocol**: Any accidental PII exposure in logs or S3 must be reported "
     "to the Privacy team within 24 hours."),
    ("runbook_airflow_restart.md",
     "# Airflow Scheduler Restart Runbook\n\n"
     "## When to Use\n"
     "Use when the Airflow scheduler is unresponsive (UI shows all DAGs as 'no status'), "
     "or when DAG runs are stuck in 'queued' for more than 20 minutes.\n\n"
     "## Steps\n"
     "```bash\n"
     "# Identify the scheduler pod (MWAA)\n"
     "aws mwaa get-environment --name prod-airflow --query 'Environment.Status'\n\n"
     "# Trigger a scheduler restart (MWAA does a rolling restart)\n"
     "aws mwaa create-web-login-token --name prod-airflow\n"
     "# Navigate to Admin -> Configuration -> Restart Scheduler\n"
     "```\n\n"
     "## Expected Recovery Time\n"
     "MWAA scheduler restart: 3-5 minutes. DAGs in 'queued' state will resume "
     "automatically after restart. Do NOT manually clear the queue during restart."),
]

print(f"Baseline ops supervisor ready.")
print(f"Pipeline ops corpus loaded: {len(PIPELINE_OPS_CORPUS)} documents.")

# Section 1: Chunking for Data Engineering

## Why Runbooks Break Fixed-Size Chunking

Fraud policies are dense paragraphs -- a 300-token chunk usually captures a complete
rule. Runbooks are different:

- **Structured sections**: `## Symptoms`, `## Root Cause`, `## Recovery Steps`
- **Code blocks**: multi-line bash sequences that MUST stay together to be useful
- **Numbered procedures**: step 2 only makes sense after step 1

A 300-token fixed-size chunk will cut a bash code block in half. The agent retrieves
half a recovery procedure and gives an incomplete (or dangerous) answer.

**Small chunks (300 tokens)**: Recovery Steps may be split. The agent sees Step 1 but
not Step 2. It gives partial guidance. For on-call recovery at 3 AM, that is dangerous.

**Large chunks (1500 tokens)**: The full recovery procedure fits in one chunk. The agent
has the complete context.

## Three Configs We Will Compare

| Config | chunk_size | chunk_overlap | Best for |
|--------|-----------|---------------|---------|
| A (small) | 300 | 30 | Short catalog cards, simple facts |
| B (medium) | 512 | 80 | General documents (benchmark default) |
| C (large) | 1500 | 200 | Runbooks with code blocks (DE default) |

The key DE finding: config C wins on runbook queries because it keeps the full
recovery procedure in one chunk. Config A destroys runbooks.

`RecursiveCharacterTextSplitter` respects paragraph -> line -> word boundaries.
For runbooks with code blocks, it tries to split at the paragraph boundary (the `##`
section header) before falling back to character-level splits. Config C is large
enough that it rarely needs to fall back.

In [ ]:
# =============================================================================
# DEMO: Split the Pipeline Ops Corpus Three Ways and Index Locally
# =============================================================================

documents = [Document(page_content=text, metadata={"source": name})
             for name, text in PIPELINE_OPS_CORPUS]

splitter_a = RecursiveCharacterTextSplitter(chunk_size=300,  chunk_overlap=30)
splitter_b = RecursiveCharacterTextSplitter(chunk_size=512,  chunk_overlap=80)
splitter_c = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)

chunks_a = splitter_a.split_documents(documents)
chunks_b = splitter_b.split_documents(documents)
chunks_c = splitter_c.split_documents(documents)

print(f"Config A (300/30):    {len(chunks_a)} chunks")
print(f"Config B (512/80):    {len(chunks_b)} chunks")
print(f"Config C (1500/200):  {len(chunks_c)} chunks")

# Build FAISS dense indexes (Titan V2 embeddings)
index_a = FAISS.from_documents(chunks_a, bedrock_embeddings)
index_b = FAISS.from_documents(chunks_b, bedrock_embeddings)
index_c = FAISS.from_documents(chunks_c, bedrock_embeddings)

print("\nThree FAISS indexes built with Titan V2 embeddings.")

In [ ]:
# =============================================================================
# DEMO: Query Each Config with Two Representative DE Queries
# =============================================================================
# Q1 is a short catalog lookup - small chunks work fine.
# Q2 is a multi-step runbook recovery - large chunks win because they keep
# the full bash sequence together instead of splitting mid-procedure.

queries = [
    ("Q1 catalog lookup",   "What is the SLA for the customers dataset?"),
    ("Q2 runbook recovery", "What are the exact steps to recover from a schema validation failure?"),
]

for label, q in queries:
    print(f"\n=== {label}: {q!r} ===")
    for name, idx in [("A (300/30)",   index_a),
                      ("B (512/80)",   index_b),
                      ("C (1500/200)", index_c)]:
        hits = idx.similarity_search_with_score(q, k=1)
        doc, score = hits[0]
        preview = doc.page_content[:150].replace("\n", " ")
        print(f"  {name:15s} score={score:6.3f}  source={doc.metadata['source']}")
        print(f"                  text: {preview}...")

print("\nObservation: config C wins on Q2 because the bash recovery steps stay together.")
print("Config A may split the bash block, losing the full recovery sequence.")

## Lab 1: Chunking Trade-off Table for DE Queries (15 min)

### Your Task

Build a pandas DataFrame comparing how the THREE chunk configs perform on FOUR
data-engineering test queries. Your table should show WHERE the large-chunk config wins.

### Steps

1. Define 4 test queries (one in each category):
   - 1 catalog query (dataset schema or SLA lookup)
   - 1 runbook query (multi-step recovery procedure)
   - 1 governance query (PII or retention policy)
   - 1 adversarial query (something the corpus likely does not cover)
2. For each (query, config) pair, retrieve top-1. Record: config label, query label,
   source filename, similarity score.
3. Build a DataFrame with columns: `config`, `query_label`, `source`, `score`.
   Sort by `query_label`.
4. Identify which query shape benefits MOST from config C, and add a one-line comment.

### Expected Output

A 12-row DataFrame (4 queries x 3 configs). Config C should win on the runbook query.

### Stretch

Add a 4th config: `MarkdownHeaderTextSplitter` using `##` as the split boundary
(all 8 corpus docs use `##` section headers). Does it produce better top-1 results
on the runbook query? Explain why or why not in a comment.

### Homework Extension

Re-ingest the runbook content into a second Bedrock KB with `HIERARCHICAL` chunking
(parent 1500 tokens, child 300 tokens). Compare hierarchical Bedrock retrieval vs
your local config C results. Why might Bedrock's hierarchical retrieval score
differently for the same text?

In [ ]:
# Lab 1: Chunking trade-off table for DE queries
# -----------------------------------------------------------------------
lab1_queries = None  # YOUR CODE - list of (label, query_string) tuples

lab1_df = None  # YOUR CODE - DataFrame with columns: config, query_label, source, score
print(lab1_df)

> **Think About It #1**: Your Lab 1 table shows config C wins on runbook queries but
> loses on simple catalog lookups. In production you cannot deploy three retrievers -
> you deploy one. For a data-platform on-call tool, which failure is more dangerous:
> getting half a recovery procedure (config A on a runbook query), or retrieving a
> slightly less relevant catalog card (config C on a schema lookup)? How would you
> document this tradeoff in an architecture decision record?

# Section 2: Reranking the RunbookRetriever

## The Bi-encoder vs Cross-encoder Gap

The `retrieve_with_filter` function from Week 17 is a **bi-encoder**: it embeds the
query once and computes cosine similarity against pre-embedded chunks. Fast, but it may
return a generic recovery procedure when you want the specific one for schema validation
failures.

A **listwise LLM reranker** (Claude Haiku) is a **cross-encoder alternative**: all
candidate chunks are sent to Haiku in one `converse` call with a ranking instruction.
Haiku scores each (query, chunk) pair jointly and returns a ranked JSON array. For
runbook queries, this matters because the difference between "schema validation failure
recovery" and "partition missing recovery" is subtle at the embedding level but obvious
when the full text is read together.

```
Query --> retrieve_with_filter (doc_type=runbook, top 10) --> Haiku listwise reranker --> top 3
```

Published guidance: reranking adds 10-30% precision for 50-100 ms per query.
For on-call tooling where one wrong recovery step can cascade failures, that lift
is worth the latency.

Note: a dedicated cross-encoder like Cohere Rerank 3.5 on Bedrock would typically
outperform an LLM prompted for ranking on domain-specific text. The listwise LLM
approach is a solid production pattern that teaches HOW reranking works. You can swap
in a dedicated model later by changing only the `bedrock_rerank` function body - the
rest of the pipeline is unchanged.

## Why Call bedrock_agent_runtime.retrieve() Directly?

`retrieve_with_filter` returns a list of dicts (structured). The reranker needs
a plain list of strings. So for reranking: call `bedrock_agent_runtime.retrieve()`
directly to get raw results, extract the `content.text` strings, pass those to
`bedrock_rerank()`.

```python
# Pattern for reranking
raw     = bedrock_agent_runtime.retrieve(knowledgeBaseId=..., ...)
texts   = [r["content"]["text"] for r in raw["retrievalResults"]]
ranked  = bedrock_rerank(query_text=q, text_sources=texts, num_results=3)
# Use ranked[i]["index"] to map back to original texts
```

In [ ]:
# =============================================================================
# DEMO: LLM-Based Listwise Reranking on Runbook Chunks (Claude Haiku)
# =============================================================================
# We use Claude Haiku as a cross-encoder alternative. All candidate chunks
# are sent in ONE converse call with a ranking prompt. Haiku returns a JSON
# array of {index, relevance_score} pairs - same output shape as a
# dedicated reranker model. No marketplace subscription required.
#
# Why this matters for runbooks: the bi-encoder's cosine similarity may score
# two different runbooks similarly (both use "recovery steps", "bash", "ETL").
# The listwise LLM sees the full (query, chunk) pair and ranks the specifically
# relevant runbook first.

import json as _json

def bedrock_rerank(query_text: str, text_sources: list, num_results: int = 3) -> list:
    """Rerank text_sources by relevance to query_text using Claude Haiku listwise ranking.

    All candidates are sent in one converse call. Haiku returns a ranked
    JSON array. Same output shape as a dedicated reranker endpoint:
      [{index, relevance_score, text}, ...] sorted by relevance_score desc.

    Args:
        query_text:   The query to rank against.
        text_sources: List of plain-text chunk strings to rank.
        num_results:  How many top results to return.

    Returns:
        List of dicts with keys: index (0-indexed into text_sources),
        relevance_score (0.0-1.0), text.
    """
    if not text_sources:
        return []

    # Cap at 10 candidates to stay within prompt budget
    sources_to_rank = text_sources[:10]
    sources_text = "\n".join(
        f"{i+1}. {src[:300]}..." if len(src) > 300 else f"{i+1}. {src}"
        for i, src in enumerate(sources_to_rank)
    )

    prompt = (
        "You are a relevance ranking expert. Given a query and a list of "
        "documents, rank them by relevance to the query.\n\n"
        f"Query: {query_text}\n\n"
        f"Documents:\n{sources_text}\n\n"
        "Return a JSON array with ALL documents ranked from most to least "
        "relevant. Include the original document number (1-indexed) and a "
        "relevance score from 0.0 to 1.0.\n\n"
        "Example format:\n"
        '[{"index": 2, "relevance_score": 0.95}, '
        '{"index": 1, "relevance_score": 0.72}, '
        '{"index": 3, "relevance_score": 0.40}]\n\n'
        "Return ONLY a valid JSON array, no other text."
    )

    try:
        response = bedrock_runtime.converse(
            modelId=MODEL_ID,
            messages=[{"role": "user", "content": [{"text": prompt}]}],
            inferenceConfig={"maxTokens": 500, "temperature": 0.0},
        )
        response_text = response["output"]["message"]["content"][0]["text"]
        ranked = _json.loads(response_text)

        result = []
        for item in ranked:
            if isinstance(item, dict) and "index" in item:
                idx = item["index"] - 1  # convert 1-indexed -> 0-indexed
                if 0 <= idx < len(sources_to_rank):
                    result.append({
                        "index":           idx,
                        "relevance_score": float(item.get("relevance_score", 0.0)),
                        "text":            sources_to_rank[idx],
                    })

        return sorted(result, key=lambda x: x["relevance_score"], reverse=True)[:num_results]

    except (_json.JSONDecodeError, KeyError, TypeError):
        print("bedrock_rerank: JSON parse failed, returning original order.")
        return [
            {"index": i, "relevance_score": 1.0 - (i * 0.1), "text": s}
            for i, s in enumerate(sources_to_rank[:num_results])
        ]


# Demo: retrieval + listwise reranking on one runbook query
q = "How do I recover from a schema validation failure in the customers ETL?"

raw = bedrock_agent_runtime.retrieve(
    knowledgeBaseId=STRANDS_DE_KNOWLEDGE_BASE_ID,
    retrievalQuery={"text": q},
    retrievalConfiguration={
        "vectorSearchConfiguration": {
            "numberOfResults": 8,
            "filter": {"equals": {"key": "doc_type", "value": "runbook"}},
        },
    },
)
candidates = [i.get("content", {}).get("text", "") for i in raw.get("retrievalResults", [])]

print(f"Retrieved {len(candidates)} runbook chunks. Reranking...\n")
ranked = bedrock_rerank(q, candidates, num_results=3)
print("Top-3 after Haiku listwise reranking:")
for i, r in enumerate(ranked, 1):
    print(f"  #{i}  score={r['relevance_score']:.3f}")
    print(f"       text: {r['text'][:160].replace(chr(10), ' ')}...")

## Lab 2: Reranked Runbook Retriever (15 min)

### Your Task

Write `reranked_runbook_retrieve` -- a function that adds Cohere Rerank 3.5 on top of
the existing Bedrock retrieval for runbook queries.

### Steps

1. Write `reranked_runbook_retrieve(query, k_retrieve=10, k_final=3)` that:
   - Calls `bedrock_agent_runtime.retrieve()` directly with `doc_type=runbook` filter
   - Extracts chunk texts from `retrievalResults` (use `r["content"]["text"]`)
   - Calls `bedrock_rerank()` to reorder
   - Returns the top `k_final` reranked results
2. Test on TWO runbook queries where you expect reranking to help.
3. For each query: print the top-1 result BEFORE and AFTER reranking.

### Expected Output

Two before/after comparisons showing which runbook chunk moves to top-1 after reranking.

### Stretch

Time the reranked call with `time.perf_counter`. For an on-call tool that runs 24/7,
how many extra milliseconds per incident is acceptable? Compare to the cost of a wrong
recovery procedure (time to detect + re-run the pipeline).

### Homework Extension

Replace `retrieve_runbook` in `runbook_retriever_agent` with a custom `@tool` that
calls `reranked_runbook_retrieve`. Re-test the agent on the same recovery question.
Does the agent's answer change? Does it become more specific?

In [ ]:
# Lab 2: Reranked runbook retriever
# -----------------------------------------------------------------------
# IMPORTANT: call bedrock_agent_runtime.retrieve() directly with doc_type=runbook
# filter. Do NOT use retrieve_with_filter (it returns structured dicts;
# bedrock_rerank() needs a plain list of strings).

def reranked_runbook_retrieve(query: str, k_retrieve: int = 10, k_final: int = 3) -> list:
    """Retrieve runbook chunks from DE KB, then rerank with Cohere 3.5."""
    # YOUR CODE
    pass


lab2_queries = None  # YOUR CODE - list of 2 runbook queries to test

for q in lab2_queries:
    # YOUR CODE - print before/after reranking for each query
    pass

In [ ]:
# Lab 2 safety-net: run this cell ONLY if you did not finish Lab 2.
# It defines a working reranked_runbook_retrieve so Lab 3 can use it.
# If you DID finish Lab 2, SKIP this cell.
# SAFETY-NET

def _sn_reranked_runbook_retrieve(query: str, k_retrieve: int = 10, k_final: int = 3) -> list:
    raw = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=STRANDS_DE_KNOWLEDGE_BASE_ID,
        retrievalQuery={"text": query},
        retrievalConfiguration={
            "vectorSearchConfiguration": {
                "numberOfResults": k_retrieve,
                "filter": {"equals": {"key": "doc_type", "value": "runbook"}},
            },
        },
    )
    items      = raw.get("retrievalResults", [])
    candidates = [i.get("content", {}).get("text", "") for i in items]
    sources    = [i.get("location", {}) for i in items]
    ranked     = bedrock_rerank(query, candidates, num_results=k_final)
    for r in ranked:
        r["source"] = sources[r["index"]]
    return ranked


_needs_net = False
try:
    if reranked_runbook_retrieve is None:
        _needs_net = True
    else:
        _probe = reranked_runbook_retrieve("test", k_retrieve=1, k_final=1)
        if _probe is None:
            _needs_net = True
except Exception:
    _needs_net = True

if _needs_net:
    reranked_runbook_retrieve = _sn_reranked_runbook_retrieve
    print("Safety-net active - using working reranked_runbook_retrieve.")
else:
    print("Lab 2 completed - keeping your reranked_runbook_retrieve.")

> **Think About It #2**: Reranking adds 50-100 ms per runbook query. In a fully
> automated on-call tool that pages the DE team and suggests a recovery action, that
> delay is invisible. But in a HUMAN on-call flow where the DE is already stressed,
> every second counts. How would you decide whether to enable reranking by default or
> make it a query-level flag? Who in your organization would have the authority to
> change that default after deployment?

# Section 3: RAGAS Evaluation + PipelineOpsAdvisor

## From "It Looks Right" to Numbers

The Week 17 ops supervisor answers runbook questions. But "it gave an answer" and "it
gave the RIGHT answer" are different. RAGAS gives you two fast metrics:

| Metric | What it asks | When it fails for on-call RAG |
|--------|--------------|-------------------------------|
| `faithfulness` | Is the answer supported by what was retrieved? | Agent added steps not in the runbook |
| `answer_relevancy` | Does the answer address the question? | Agent answered a related but different question |

For data ops: a faithfulness failure means the agent invented a recovery step. That
can cascade a failure instead of fixing it.

## PipelineOpsAdvisor: The Week 18 DE Main Outcome

After scoring the baseline, you will build `pipeline_ops_advisor` -- the Week 17
`ops_supervisor` extended with:
1. `reranked_runbook_retrieve` (from Lab 2) as a `@tool` instead of the plain runbook filter
2. RAGAS-validated quality on the 3 DE questions

The advisor is the hand-off to Week 19 (MLflow will track the RAGAS scores) and
Weeks 21-22 (Airflow will schedule periodic re-evaluation runs as the KB grows).

## RAGAS Setup

RAGAS 0.4.x uses `LangchainLLMWrapper(ChatBedrockConverse(...))` as the judge LLM.
The `evaluator_llm` and `evaluator_embeddings` are already configured in Cell 3.
Call pattern:

```python
from ragas import evaluate, EvaluationDataset, SingleTurnSample
sample  = SingleTurnSample(user_input=q, response=answer, retrieved_contexts=[...])
dataset = EvaluationDataset(samples=[sample])
result  = evaluate(dataset=dataset, metrics=[faithfulness, answer_relevancy],
                   llm=evaluator_llm, embeddings=evaluator_embeddings)
```

In [ ]:
# =============================================================================
# DEMO: Score One Runbook Answer with RAGAS
# =============================================================================

q = "What are the recovery steps for a schema validation failure in the customers ETL?"

# Retrieve from DE KB (runbook filter, 3 chunks)
raw = bedrock_agent_runtime.retrieve(
    knowledgeBaseId=STRANDS_DE_KNOWLEDGE_BASE_ID,
    retrievalQuery={"text": q},
    retrievalConfiguration={
        "vectorSearchConfiguration": {
            "numberOfResults": 3,
            "filter": {"equals": {"key": "doc_type", "value": "runbook"}},
        },
    },
)
retrieved_contexts = [i.get("content", {}).get("text", "")
                      for i in raw.get("retrievalResults", [])]

# Generate answer from retrieved context only
gen_prompt = (
    "Answer using only the context. Be specific about commands and steps.\n\n"
    f"Context:\n{chr(10).join(retrieved_contexts)}\n\nQuestion: {q}\nAnswer:"
)
answer = langchain_llm.invoke(gen_prompt).content

# RAGAS evaluation
sample  = SingleTurnSample(
    user_input=q,
    response=answer,
    retrieved_contexts=retrieved_contexts,
)
dataset = EvaluationDataset(samples=[sample])
result  = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
)
print("RAGAS result for demo question:")
print(result)
print(f"\nAnswer preview: {answer[:300]}...")

## Lab 3: RAGAS Score Sheet + PipelineOpsAdvisor (15 min)

### Your Task (two parts)

**Part A** (10 min): Score 3 DE questions with RAGAS (baseline vs reranked).
**Part B** (5 min): Build the `pipeline_ops_advisor` that uses `reranked_runbook_retrieve`.

### Part A Steps

1. Define 3 DE questions:
   - 1 runbook recovery question
   - 1 SLA or escalation question
   - 1 PII or governance question
2. Write `build_de_sample(question, use_rerank=False) -> SingleTurnSample`:
   - If `use_rerank=False`: retrieve from DE KB with runbook filter, generate answer
   - If `use_rerank=True`: use `reranked_runbook_retrieve`, generate answer
   - Both paths must return a `SingleTurnSample`
3. Score BOTH configs on all 3 questions with `[faithfulness, answer_relevancy]`.
4. Print a DataFrame: question, config, faithfulness, answer_relevancy.

### Part B Steps

5. Define `retrieve_runbook_reranked` as a `@tool` that calls `reranked_runbook_retrieve`.
6. Build `pipeline_ops_advisor = Agent(...)` using that tool.
7. Test with one recovery question and verify it answers from the KB.

### Expected Output

- A 6-row DataFrame (3 questions x 2 configs) with RAGAS scores
- `pipeline_ops_advisor` successfully answering a runbook query

### Stretch

Add a `latency_ms` column to the DataFrame by timing `build_de_sample` with
`time.perf_counter`. Does the reranked config add more latency than expected?

### Homework Extension

Run the same 3 test queries from Week 17 DE Lab 3 on your new `pipeline_ops_advisor`.
Does the supervisor's answer quality improve? Write one paragraph on whether the
RAGAS offline lift survives the full agent loop.

In [ ]:
# Lab 3: RAGAS score sheet + PipelineOpsAdvisor
# -----------------------------------------------------------------------
# Part A: build_de_sample retrieves directly from the DE KB
# (not via strands_tools.retrieve - RAGAS needs plain text strings)

def build_de_sample(question: str, use_rerank: bool = False) -> SingleTurnSample:
    """Retrieve + generate + wrap into a SingleTurnSample for RAGAS."""
    # YOUR CODE
    pass


lab3_questions = None  # YOUR CODE - list of 3 DE questions (strings)

lab3_df = None  # YOUR CODE - DataFrame with columns: question, config, faithfulness, answer_relevancy
print(lab3_df)


# -----------------------------------------------------------------------
# Part B: PipelineOpsAdvisor

@tool
def retrieve_runbook_reranked(query: str) -> str:
    """Retrieve runbook chunks reranked by Cohere 3.5 for on-call recovery.

    Args:
        query: The ops scenario or failure description.
    """
    # YOUR CODE
    pass


pipeline_ops_advisor = None  # YOUR CODE - Agent using retrieve_runbook_reranked

# Summary: Week 18 DE

## Key Takeaways

### Chunking for Runbooks
- Runbooks with code blocks need larger chunks (1500/200) to keep recovery procedures
  intact. Fixed-size 300-token chunks destroy bash sequences and numbered steps.
- `RecursiveCharacterTextSplitter` respects paragraph -> line -> word boundaries,
  making it the right choice before considering more exotic splitters.
- In production, Bedrock HIERARCHICAL chunking (parent 1500, child 300) achieves
  the same outcome without a local FAISS index.

### Reranking for On-Call Tooling
- Cohere Rerank 3.5 on Bedrock (`bedrock-agent-runtime.rerank`) adds 50-100 ms
  per runbook query. For compliance-critical on-call tools, that precision lift
  is worth the latency.
- Amazon Rerank 1.0 is NOT available in us-east-1. Cohere Rerank 3.5 IS.
- Call `bedrock_agent_runtime.retrieve()` directly (not `retrieve_with_filter`)
  before reranking -- the rerank API needs plain strings, not structured dicts.

### RAGAS for Data Ops
- `faithfulness` catches the most dangerous failure: the agent invented a recovery
  step not in the runbook. Low faithfulness on a runbook query can cascade an incident.
- `answer_relevancy` catches "the agent answered a related but different question."
  For on-call, that means the engineer follows the wrong recovery path.

### PipelineOpsAdvisor
- The Week 18 DE main outcome: `ops_supervisor` from Week 17 DE, now with
  `reranked_runbook_retrieve` as the runbook tool and RAGAS-validated quality.
- Carry `pipeline_ops_advisor` and your RAGAS score DataFrames into Weeks 19-20
  for DVC versioning and MLflow tracking.

## Looking Ahead

- **Week 19 (MLOps Part 1)**: Every RAGAS score DataFrame you built this week would
  live in MLflow next week, and every eval dataset would be DVC-versioned.
- **Week 20 (MLOps Part 2)**: Online observability with Langfuse around
  `pipeline_ops_advisor`. Today's RAGAS scores are offline; Week 20's are online.
- **Weeks 21-22 (Airflow)**: Schedule periodic RAGAS re-evaluation runs as a DAG so
  your scores stay fresh as the runbook KB grows.

## Homework

### Homework 1: Hierarchical Chunking (Lab 1 extension)
Create a second Bedrock KB with `ChunkingConfiguration=HIERARCHICAL` (parent 1500,
child 300, overlap 60) and ingest the same 8 corpus documents. Compare retrieval
quality on the runbook query vs your local config C index. Why might they score
differently even for identical text?

### Homework 2: Reranker Latency Profiling (Lab 2 extension)
Profile `reranked_runbook_retrieve` with `time.perf_counter`. For an on-call tool
that runs 3 retrieve calls per incident and handles 20 incidents/day, what is the
daily wall-clock cost of reranking? Is there a query complexity threshold where
the latency becomes unacceptable?

### Homework 3: CI Gate (Lab 3 extension)
Write a `gate(df, min_faith=0.8, min_rel=0.7)` function that returns `True` only
if every row in your Lab 3 RAGAS DataFrame passes both thresholds. This is the
deployment gate for `pipeline_ops_advisor`. Which threshold matters more for
on-call tooling, and why?